# Water Hydrants and Candidate Swimming Pools – GIS Workflow

## Project Objective

The aim of this project is to build a **reproducible GIS workflow** that detects candidate swimming pools from orthophotos, combines them with existing fire hydrant data, and estimates the potential improvement in water-access coverage for selected villages in Carinthia, Austria.

## Workflow

The notebook consists of the following steps:

1. **Configuration**

   * Define coordinate reference systems (CRS), study areas (bounding boxes), and file paths.

2. **Data Acquisition**

   * Download fire hydrant locations from OpenStreetMap (OSM).
   * Download orthophoto tiles for each study area.

3. **Orthophoto Preparation**

   * Merge orthophoto tiles into a single raster for each village.
   * Clip the merged raster to the corresponding study area.

4. **Swimming Pool Detection**

   * Apply a pre-trained YOLO model to detect candidate swimming pools from the orthophotos.
   * Export detected pools as GeoPackage files.

5. **Coverage Analysis**

   * Create 150 m buffer zones around fire hydrants and candidate pools.
   * Calculate the current hydrant coverage and the potential coverage gain when candidate pools are considered as additional water sources.

6. **Visualization**

   * Generate interactive web maps displaying:

     * Fire hydrants
     * Candidate swimming pools
     * Buffer zones
     * Estimated coverage gain

## Input Data

* OpenStreetMap fire hydrants
* Orthophotos provided by the State of Carinthia
* Bounding boxes defining the study areas

## Output

The workflow produces:

* Fire hydrant datasets (GeoPackage)
* Merged and clipped orthophotos
* Candidate swimming pool datasets (GeoPackage)
* Coverage analysis results (.csv)
* Interactive web maps (.html)

The notebook is designed to be reproducible by modifying only the parameters in the **Configuration** section.


In [ ]:
#!pip install rasterio pyproj requests tqdm geopandas requests duckdb osmnx
#!pip install ultralytics huggingface_hub

In [1]:
#import libraries

import os
from pathlib import Path

# data handling
import osmnx as ox
import requests
import geopandas as gpd
import pandas as pd
import numpy as np 

#AI
from ultralytics import YOLO
from huggingface_hub import hf_hub_download 
from rasterio.windows import Window 
from PIL import Image

# geo
from shapely.geometry import box
from rasterio.mask import mask
import rasterio
from rasterio.merge import merge
from shapely.geometry import box

# visualization
import folium


Directory structure is defined: Data and Output directory are made, if the don't exist.

In [2]:
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./outputs")
RAW_TILES_DIR = Path("./data/raw_tiles")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

### CONFIG - SECTION
Change parameters (bbox coordinates, tile urls, crs, output directory) to individual needs in the User Configuration.

In [3]:
CONFIG = {
    # CRS:
    # Target CRS for the script (all outputs will be in this CRS, except the folium webmap)
    "TARGET_CRS": "EPSG:31255",
    "WEBMAP_CRS": "EPSG:4326",

    #Buffer Radius in Meter
    "BUFFER_RADIUS": 150,

    #bbox-extents coordinates {bbox_name: [minx, miny, maxx, maxy]}
    "bboxes": {
        "bbox_lind_velden": [50051.4148721720994217,162183.1916923190001398 , 52118.7226579226989998,164240.5835017400095239],
        "bbox_hermagor": [2322.1855355381098889,164526.2229417290072888 , 3819.7314718246998382,165894.6583633570116945],
        "bbox_treffen": [39421.6747735511016799,169668.5483137849951163 , 40566.7011035258983611,170817.9372140899940860],
    },

    # Tiles for each bbox: {bbox_name: [tile_url1, tile_url2, ...]}
    "tiles_per_bbox": {
        "bbox_lind_velden": [
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5117-41.tif", "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5117-42.tif", 
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5117-49.tif","https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5117-50.tif",
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5117-57.tif","https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5117-58.tif",  

        ],
        "bbox_hermagor": [
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4617-35.tif","https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4617-36.tif",
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4617-44.tif","https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4617-43.tif",
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4617-42.tif","https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4617-34.tif",                     

        ],
        "bbox_treffen": [
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5018-73.tif", "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4918-80.tif", 
            "https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/4917-08.tif","https://gis.ktn.gv.at/ogd/CUSTOM/ORTHO/rgbi/5017-01.tif",
        ],
    },

}

#configure hydrant extraction
place_name1 = "Velden, Austria" #change to desired place
output_file1 = DATA_DIR/"fire_hydrants_velden.gpkg" #change to desired output file path

place_name2 = "Treffen, Austria"  #change to desired place
output_file2 = DATA_DIR/"fire_hydrants_treffen.gpkg" #change to desired output file path

place_name3 = "Hermagor, Austria"  #change to desired place
output_file3 = DATA_DIR/"fire_hydrants_hermagor.gpkg" #change to desired output file path


### STEP 1: Download the Fire Hydrantes from OSM

In [4]:
def download_hydrants(place_name, bbox, output_file):

    #This Function: 
    #   1. Downloads the fire hydrants from OSM 
    #   2. Reproject the downloaded file to the Target_CRS
    #   3. clip the download to the bbox
    #
    #place_name: Use the name of the Town/City you want to download
    #input 'bbox' as list of [minx, miny, maxx, maxy]:  (example: CONFIG["bboxes"]["bbox_hermagor"])
    #output_file: Use the directory and name, how you want to store the downloaded data

    # Download part:
    if os.path.exists(output_file):
        print(f"File {output_file} already exists. Skipping download and save.")
    else:
        # Download fire hydrants for the area
        tags = {"emergency": "fire_hydrant"}
        gdf1 = ox.features_from_place(place_name, tags=tags)

        # Check if data was found
        if gdf1.empty:
            print("No fire hydrants found for the specified area.")
        else:
            # Save to GeoPackage
            gdf1.to_file(output_file, driver="GPKG")
            print(f"Fire hydrants saved to {output_file}")
    
    # Reprojection part:
    if os.path.exists(output_file):
        gdf = gpd.read_file(output_file)
        gdf = gdf.to_crs(CONFIG["TARGET_CRS"])
        gdf.to_file(output_file, driver="GPKG")
        print(f"Reprojected {output_file} to {CONFIG["TARGET_CRS"]}")

    # Clipping part:
    gdf = gpd.read_file(output_file)
    bbox = box(*bbox)
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs=CONFIG["TARGET_CRS"])
    gdf_clipped = gpd.clip(gdf, bbox_gdf)
    gdf_clipped.to_file(output_file, driver="GPKG")
    print(f"Clipped file save to {output_file}")


In [5]:
#Download for Hydrants for Velden
download_hydrants(place_name1, CONFIG["bboxes"]["bbox_lind_velden"], output_file1)

File data\fire_hydrants_velden.gpkg already exists. Skipping download and save.
Reprojected data\fire_hydrants_velden.gpkg to EPSG:31255
Clipped file save to data\fire_hydrants_velden.gpkg


In [6]:
#Download for Hydrants for Treffen
download_hydrants(place_name2, CONFIG["bboxes"]["bbox_treffen"], output_file2)

File data\fire_hydrants_treffen.gpkg already exists. Skipping download and save.
Reprojected data\fire_hydrants_treffen.gpkg to EPSG:31255
Clipped file save to data\fire_hydrants_treffen.gpkg


In [7]:
#Download for Hydrants for Hermagor
download_hydrants(place_name3, CONFIG["bboxes"]["bbox_hermagor"], output_file3)

File data\fire_hydrants_hermagor.gpkg already exists. Skipping download and save.
Reprojected data\fire_hydrants_hermagor.gpkg to EPSG:31255
Clipped file save to data\fire_hydrants_hermagor.gpkg


### STEP 2: Download Orthophotos 
For every village .TIFF tiles are download from the Land Kärnten server and stored in the `RAW_TILES` folders.

In [8]:
# Extract config
tiles_per_bbox = CONFIG["tiles_per_bbox"]  # Fixed: Use "tiles_per_bbox" instead of "tile_urls"


# Download tiles
for bbox_name, tile_urls in tiles_per_bbox.items():
    # create a new bbox_dir for every villages
    bbox_dir = os.path.join(RAW_TILES_DIR, bbox_name)
    os.makedirs(bbox_dir, exist_ok=True)

    for url in tile_urls:  
        # extract filename and build local path
        filename = os.path.basename(url)  
        filepath = os.path.join(bbox_dir, filename)

        if not os.path.exists(filepath): #when data isn't yet exist, download it.
            try:
                response = requests.get(url, stream=True, timeout=10) #Download request
                response.raise_for_status()  # Check for HTTP errors
                with open(filepath, "wb") as f: #Save TIFF file
                    f.write(response.content)
                print(f"Downloaded: {filename}")
            except requests.exceptions.RequestException as e:
                print(f"Failed to download {url}: {e}")

In [9]:
#check file size for realistic sizes - if maybe a download was incomplete
for bbox_name in ["bbox_lind_velden", "bbox_hermagor", "bbox_treffen"]:
    bbox_dir = os.path.join(RAW_TILES_DIR, bbox_name)
    print(f"\nFiles in {bbox_dir}:")
    for file in os.listdir(bbox_dir):
        filepath = os.path.join(bbox_dir, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)  # Size in MB
        print(f"  - {file}: {size_mb:.2f} MB")


Files in data\raw_tiles\bbox_lind_velden:
  - 5117-41.tif: 34.79 MB
  - 5117-42.tif: 34.59 MB
  - 5117-49.tif: 36.33 MB
  - 5117-50.tif: 35.58 MB
  - 5117-57.tif: 37.20 MB
  - 5117-58.tif: 38.13 MB

Files in data\raw_tiles\bbox_hermagor:
  - 4617-34.tif: 35.87 MB
  - 4617-35.tif: 38.59 MB
  - 4617-36.tif: 36.26 MB
  - 4617-42.tif: 33.41 MB
  - 4617-43.tif: 35.76 MB
  - 4617-44.tif: 35.93 MB

Files in data\raw_tiles\bbox_treffen:
  - 4917-08.tif: 38.18 MB
  - 4918-80.tif: 35.99 MB
  - 5017-01.tif: 37.15 MB
  - 5018-73.tif: 39.52 MB


### Step 3: Merge Orthophoto Tiles

Goal:
Combine all tiles of a village into a single raster.

Output:
- one TIFF-File per village

In [10]:
def merging_tiles(input_dir, output_file):

    # The Funktion Merge the tiles which are in the input_dir to one tiff-file
    
    # for the input_dir choose:
    # the folder with the clipped TIFFs
    # for example: input_dir = RAW_TILES_DIR/"bbox_hermagor"

    # output_file is a:
    # Output mosaic
    # for example: output_file = OUTPUT_DIR/"hermagor_merged.tif"


    # Open all TIFF files in the folder
    src_files = []

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(".tif"):
            filepath = os.path.join(input_dir, filename)
            src_files.append(rasterio.open(filepath))

    # Merge
    mosaic, out_transform = merge(src_files)

    # Metadata from first raster
    out_meta = src_files[0].meta.copy()

    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "compress": "lzw"
    })

    # Write merged raster
    with rasterio.open(output_file, "w", **out_meta) as dest:
        dest.write(mosaic)

    # Close files
    for src in src_files:
        src.close()

    print(f"Mosaic saved to: {output_file}")

In [11]:
#Merge for Hermagor
merging_tiles(RAW_TILES_DIR/"bbox_hermagor", DATA_DIR/"hermagor_merged.tif")

Mosaic saved to: data\hermagor_merged.tif


In [12]:
#Merge for Lind ob Velden
merging_tiles(RAW_TILES_DIR/"bbox_lind_velden", DATA_DIR/"lind_velden_merged.tif")

Mosaic saved to: data\lind_velden_merged.tif


In [13]:
#Merge for Treffen
merging_tiles(RAW_TILES_DIR/"bbox_treffen", DATA_DIR/"treffen_merged.tif")

Mosaic saved to: data\treffen_merged.tif


### Step 4: Clipping of the merged Orthofotos

The merged Orthofotos are clipped to the study area (`bboxes`) of every village

In [14]:

def clipping_bbox(merged_orthofoto, bbox, output_path):
    #Function does a Clipping of the merged_orthofoto with a bbox 
    #
    # input file: merged_orthofoto(example: hermagor_merged.tif)
    # input bbox as list of [minx, miny, maxx, maxy]:  (example: CONFIG["bboxes"]["bbox_hermagor"])
    # output_path: path and name of the output file (example: hermagor_clipped.tif)
    
    bbox_polygon = box(*bbox)
    # Creates a polygon

    with rasterio.open(merged_orthofoto) as src:
        # Ensure the tile CRS matches the bbox CRS (EPSG:31255)
        if str(src.crs) != "EPSG:31255":
            print(f"Warning: Tile {merged_orthofoto} CRS is {src.crs}, but bbox is in EPSG:31255")

        out_image, out_transform = mask(
            src, [bbox_polygon], crop=True, nodata=src.nodata
            )
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "compress": "lzw"
        })

        with rasterio.open(output_path, "w", **out_meta) as dst:
            dst.write(out_image)

In [15]:
clipping_bbox(DATA_DIR/"lind_velden_merged.tif", CONFIG["bboxes"]["bbox_lind_velden"], DATA_DIR/"lind_velden_clipped.tif")

In [16]:
clipping_bbox(DATA_DIR/"hermagor_merged.tif", CONFIG["bboxes"]["bbox_hermagor"], DATA_DIR/"hermagor_clipped.tif")

In [17]:
clipping_bbox(DATA_DIR/"treffen_merged.tif", CONFIG["bboxes"]["bbox_treffen"], DATA_DIR/"treffen_clipped.tif")

### STEP 5: Swimming Pool Detection

Goal:
Detect candidate swimming pools in orthophotos using a YOLO model.

Load the Swimming Pool detector (AI-Model)

In [18]:
#Load the AI model 
model_path = hf_hub_download(
    repo_id="mozilla-ai/swimming-pool-detector",
    filename="model.pt"
)
model = YOLO(model_path)

Function for Detection of swimming pools of a orthofoto

In [19]:
#Chipping + Detection Function
def detect_pools_in_tif(tif_path, chip_size=640, overlap=0.2, conf_threshold=0.3):
    # The Function detects swimming pools in a Orthofoto

    # Input is a orthofoto: tif_path
    # Output is GeoDataFrame of detected pools 
    # chips_size, overlap and conf_threshold are already set
    
    results_list = []
    stride = int(chip_size * (1 - overlap))

    with rasterio.open(tif_path) as src:
        crs = src.crs
        transform = src.transform
        width, height = src.width, src.height
        
        # only take RGB bands
        band_count = min(src.count, 3)

        for row_off in range(0, height, stride):
            for col_off in range(0, width, stride):
                # Window definieren
                win_width = min(chip_size, width - col_off)
                win_height = min(chip_size, height - row_off)
                window = Window(col_off, row_off, win_width, win_height)

                # read chip 
                chip_data = src.read(
                    list(range(1, band_count + 1)),
                    window=window
                )  # shape: (bands, H, W)

                # conver to PIL 
                chip_rgb = np.moveaxis(chip_data, 0, -1)
                if chip_rgb.dtype != np.uint8:
                    chip_rgb = (chip_rgb / chip_rgb.max() * 255).astype(np.uint8)
                
                # paddle to chip size 
                if chip_rgb.shape[0] < chip_size or chip_rgb.shape[1] < chip_size:
                    padded = np.zeros((chip_size, chip_size, 3), dtype=np.uint8)
                    padded[:chip_rgb.shape[0], :chip_rgb.shape[1]] = chip_rgb
                    chip_rgb = padded

                pil_img = Image.fromarray(chip_rgb)

                # Inference
                detections = model.predict(pil_img, conf=conf_threshold, verbose=False)

                for det in detections:
                    if det.boxes is None or len(det.boxes) == 0:
                        continue
                    for bbox_px in det.boxes.xyxy.cpu().numpy():
                        # Pixel-Coordinates in Chip → in whole Raster
                        x1_img = col_off + bbox_px[0]
                        y1_img = row_off + bbox_px[1]
                        x2_img = col_off + bbox_px[2]
                        y2_img = row_off + bbox_px[3]

                        # Pixel → Geo- Coordinates
                        x1_geo, y1_geo = rasterio.transform.xy(transform, y1_img, x1_img)
                        x2_geo, y2_geo = rasterio.transform.xy(transform, y2_img, x2_img)

                        geom = box(
                            min(x1_geo, x2_geo), min(y1_geo, y2_geo),
                            max(x1_geo, x2_geo), max(y1_geo, y2_geo)
                        )
                        conf_score = float(det.boxes.conf.cpu().numpy()[0])
                        results_list.append({
                            "geometry": geom,
                            "confidence": conf_score,
                            "source_tif": os.path.basename(tif_path)
                        })

    if results_list:
        gdf = gpd.GeoDataFrame(results_list, crs=crs)
        # remove duplicates with overlapping
        gdf = gdf.drop_duplicates(subset=["geometry"])
        return gdf
    else:
        return gpd.GeoDataFrame(columns=["geometry", "confidence", "source_tif"], crs=crs)



Run the Swimming pool detection Function for all villages

In [20]:
# Do the Detection for all three villages 
mosaics = {
    "velden":   DATA_DIR/"lind_velden_clipped.tif",
    "hermagor": DATA_DIR/"hermagor_clipped.tif",
    "treffen":  DATA_DIR/"treffen_clipped.tif",
}

all_pools = {}

for village, tif_path in mosaics.items():
    gdf_pools = detect_pools_in_tif(str(tif_path))
    all_pools[village] = gdf_pools
    
    #save
    out_path = OUTPUT_DIR / f"pools_{village}.gpkg"
    gdf_pools.to_file(out_path, driver="GPKG")
    print(f"Saved to {out_path}")

Saved to outputs\pools_velden.gpkg
Saved to outputs\pools_hermagor.gpkg
Saved to outputs\pools_treffen.gpkg


### STEP 7: Anaylse the Coverage Fire Hydrants plus the added Pools

In the follwoing Section the Coverage of fire hydrants, of pools and of the combination of them is calculated

In [21]:
# BUFFER_RADIUS = 150 Meter, stored in the CONFIG

#dicitionary with .gpkg files of the hydrants and pools; polygons of the bboxes
villages = {
    "velden":   {"hydrants": DATA_DIR / "fire_hydrants_velden.gpkg",
                 "pools":    OUTPUT_DIR / "pools_velden.gpkg", 
                 "study_area": box(*CONFIG["bboxes"]["bbox_lind_velden"])},
    "hermagor": {"hydrants": DATA_DIR / "fire_hydrants_hermagor.gpkg",
                 "pools":    OUTPUT_DIR / "pools_hermagor.gpkg",
                 "study_area": box(*CONFIG["bboxes"]["bbox_hermagor"])},
    "treffen":  {"hydrants": DATA_DIR / "fire_hydrants_treffen.gpkg",
                 "pools":    OUTPUT_DIR / "pools_treffen.gpkg",
                 "study_area": box(*CONFIG["bboxes"]["bbox_treffen"])}
}


results = []

for village, data in villages.items():
    print(f"\n📍 {village.upper()}")

    # fire_hydrents
    hydrants = gpd.read_file(data["hydrants"])
    hydrants = hydrants[hydrants.geometry.geom_type == "Point"]  # nur Punkte
    hydrants = hydrants.to_crs(CONFIG["TARGET_CRS"])
    

    # pool_polygons - seting center for buffering
    pools = gpd.read_file(data["pools"])
    pools = pools.to_crs(CONFIG["TARGET_CRS"])
    pools["geometry"] = pools.geometry.centroid  # center of the bounding box
    

    #Inspection area - area calculation
    study_area_m2 = data["study_area"].area

    # coverage with fire_hydrants of the inspected area
    hydrant_buffers = hydrants.geometry.buffer(CONFIG["BUFFER_RADIUS"])
    coverage_before = hydrant_buffers.union_all()
    area_before = coverage_before.intersection(data["study_area"]).area

    # coverage with fire_hydrants + detected pools of the inspected area
    pool_buffers = pools.geometry.buffer(CONFIG["BUFFER_RADIUS"])
    coverage_after = coverage_before.union(pool_buffers.union_all())
    area_after = coverage_after.intersection(data["study_area"]).area

    # gain of the coverage
    gain_m2 = area_after - area_before
    gain_pct = (gain_m2 / study_area_m2) * 100 if study_area_m2 > 0 else 0
    coverage_before_pct = (area_before / study_area_m2) * 100
    coverage_after_pct  = (area_after  / study_area_m2) * 100

    #print the results
    print(f"  Fire Hydrants:        {len(hydrants)}")
    print(f"  Candidate Pools:      {len(pools)}")
    print(f"  Coverage before:      {coverage_before_pct:.1f}%")
    print(f"  Coverage after:       {coverage_after_pct:.1f}%")
    print(f"  Gain:                 +{gain_pct:.1f}% ({gain_m2/1e4:.2f} ha)")

    #add the results to the results dicitionary
    results.append({
        "village":              village.capitalize(),
        "n_hydrants":           len(hydrants),
        "n_pools":              len(pools),
        "coverage_before_pct":  round(coverage_before_pct, 1),
        "coverage_after_pct":   round(coverage_after_pct, 1),
        "gain_pct":             round(gain_pct, 1),
        "gain_ha":              round(gain_m2 / 1e4, 2),
        "priority":             "" #get filled later
    })


# Village Comparison Table 
df_results = pd.DataFrame(results).sort_values("gain_pct", ascending=False)

# Set priority: Highest gain --> highest priority
df_results["priority"] = ["Hoch", "Mittel", "Niedrig"][:len(df_results)]

print("\n\n VILLAGE COMPARISON TABLE")
print("=" * 65)
print(df_results[["village", "n_hydrants", "n_pools",
                   "coverage_before_pct", "coverage_after_pct",
                   "gain_pct", "priority"]].to_string(index=False))

# Save the table to a .csv file
df_results.to_csv(OUTPUT_DIR / "coverage_comparison.csv", index=False)
print(f"\n Saved to outputs/coverage_comparison.csv")


📍 VELDEN
  Fire Hydrants:        10
  Candidate Pools:      76
  Coverage before:      13.8%
  Coverage after:       49.2%
  Gain:                 +35.4% (150.53 ha)

📍 HERMAGOR
  Fire Hydrants:        5
  Candidate Pools:      31
  Coverage before:      14.0%
  Coverage after:       61.8%
  Gain:                 +47.8% (97.96 ha)

📍 TREFFEN
  Fire Hydrants:        8
  Candidate Pools:      28
  Coverage before:      28.4%
  Coverage after:       62.3%
  Gain:                 +34.0% (44.69 ha)


 VILLAGE COMPARISON TABLE
 village  n_hydrants  n_pools  coverage_before_pct  coverage_after_pct  gain_pct priority
Hermagor           5       31                 14.0                61.8      47.8     Hoch
  Velden          10       76                 13.8                49.2      35.4   Mittel
 Treffen           8       28                 28.4                62.3      34.0  Niedrig

 Saved to outputs/coverage_comparison.csv


### STEP 8: Visualisation of the Results

Webmaps are visualised for each village. 

Items:
- Study area
- Fire Hydrants + Buffer(150m)
- Detected Pools + Buffer(150m)
- Gain of Coverage area



In [22]:
#This Function defines the style for the gain area in the webmap
def gain_style(feature):
    return {
        "fillColor": "#FFD700",
        "color": "#FFA500",
        "weight": 1.5,
        "fillOpacity": 0.35
    }

#This Function defines the style for the total coverage area
def total_coverage_style(feature):
    return {
        "fillColor": "#7E57C2",
        "color": "#512DA8",
        "weight": 1.5,
        "fillOpacity": 0.20
    }

#This Function defines the style for the studyarea in the webmap
def study_area_style(feature):
    return {
        "fillColor": "none",
        "color": "black",
        "weight": 2
    }

#If you want to execute the Functiion in the next cell:
#You don't need feature, because it returns anyway the style

In [23]:
# WEB MAPS (with FOLIUM) and PRESENTATION

TARGET_CRS_LATLON = CONFIG["WEBMAP_CRS"] # folium needs WGS84

for village, data in villages.items():
    print(f"\n  Erstelle Karte für {village}...")

    # Load the Data
    hydrants = gpd.read_file(data["hydrants"])
    hydrants = hydrants[hydrants.geometry.geom_type == "Point"].to_crs(CONFIG["TARGET_CRS"])
    hydrants_ll = hydrants.to_crs(TARGET_CRS_LATLON)

    pools = gpd.read_file(data["pools"]).to_crs(CONFIG["TARGET_CRS"])
    pools["geometry"] = pools.geometry.centroid
    pools_ll = pools.to_crs(TARGET_CRS_LATLON)

    # Set the center of the map 
    center = [hydrants_ll.geometry.y.mean(), hydrants_ll.geometry.x.mean()]
    m = folium.Map(location=center, zoom_start=14, tiles="OpenStreetMap")

    # different layers
    fg_hydrants      = folium.FeatureGroup(name="🔵 Fire-Hydrants", show=True)
    fg_pools         = folium.FeatureGroup(name="🟢 Candidate Pools (AI)", show=True)
    fg_buf_hydrants  = folium.FeatureGroup(name="🔵 Fire-Hydrant-Buffer (150m)", show=False)
    fg_buf_pools     = folium.FeatureGroup(name="🟢 Pool-Buffer (150m)", show=False)
    fg_coverage      = folium.FeatureGroup(name="🟡 Coverage Gain", show=True)
    fg_coverage_tot =  folium.FeatureGroup(name="🟣 Total Coverage", show=False)
    fg_study_area   = folium.FeatureGroup(name="    Study Area", show=True)

    # hydrant buffer, pool buffer, the gain coverage (difference of pool buffer and hydrant buffer) and total overage
    hydrant_union   = hydrants.geometry.buffer(CONFIG["BUFFER_RADIUS"]).union_all()
    pool_union      = pools.geometry.buffer(CONFIG["BUFFER_RADIUS"]).union_all()
    gain_area       = pool_union.difference(hydrant_union)
    total_coverage  = pool_union.union(hydrant_union)

    
    # To a GeoDataFrame for folium
    # gain
    gain_gdf = gpd.GeoDataFrame(geometry=[gain_area], crs=CONFIG["TARGET_CRS"]).to_crs(TARGET_CRS_LATLON)
    
    # total coverage
    total_gdf = gpd.GeoDataFrame(geometry=[total_coverage], crs=CONFIG["TARGET_CRS"]).to_crs(TARGET_CRS_LATLON)
    # bbox
    study_area_polygon = data["study_area"]
    study_area_gdf = gpd.GeoDataFrame(geometry=[study_area_polygon], crs=CONFIG["TARGET_CRS"]).to_crs(TARGET_CRS_LATLON)   
    
    #add to Folium
    folium.GeoJson(gain_gdf, style_function=gain_style, tooltip="Coverage Gain durch Pools").add_to(fg_coverage)
    folium.GeoJson(total_gdf, style_function=total_coverage_style, tooltip="Total Coverage Hydrants + Pools").add_to(fg_coverage_tot)
    folium.GeoJson(study_area_gdf, style_function=study_area_style, tooltip="Study Area").add_to(fg_study_area)

    # hydrant buffer
    for index, hydrant in hydrants_ll.iterrows():
        folium.Circle(
            location=[hydrant.geometry.y, hydrant.geometry.x],
            radius=CONFIG["BUFFER_RADIUS"],
            color="#1565C0", fill=True, fill_opacity=0.08, weight=1
        ).add_to(fg_buf_hydrants)

    # pool buffer
    for index, pool in pools_ll.iterrows():
        folium.Circle(
            location=[pool.geometry.y, pool.geometry.x],
            radius=CONFIG["BUFFER_RADIUS"],
            color="#2E7D32", fill=True, fill_opacity=0.08, weight=1
        ).add_to(fg_buf_pools)

    # hydrants
    for index, hydrant in hydrants_ll.iterrows():
        folium.CircleMarker(
            location=[hydrant.geometry.y, hydrant.geometry.x],
            radius=5, color="#1565C0", fill=True,
            fill_color="#42A5F5", fill_opacity=0.9, weight=2,
            tooltip="🔵 Hydrant"
        ).add_to(fg_hydrants)

    # pool
    for index, pool in pools_ll.iterrows():
        folium.CircleMarker(
            location=[pool.geometry.y, pool.geometry.x],
            radius=5, color="#2E7D32", fill=True,
            fill_color="#66BB6A", fill_opacity=0.9, weight=2,
            tooltip=f"🟢 Pool (conf: {pool.get('confidence', '?'):.2f})"
        ).add_to(fg_pools)

    # All Layer add to the map
    fg_coverage_tot.add_to(m)
    fg_coverage.add_to(m)
    fg_buf_hydrants.add_to(m)
    fg_buf_pools.add_to(m)
    fg_hydrants.add_to(m)
    fg_pools.add_to(m)
    fg_study_area.add_to(m)
   
    # Legend
    legend_html = f"""
    <div style="position:fixed; bottom:30px; left:30px; z-index:1000;
                background:white; padding:12px 16px; border-radius:8px;
                border:1px solid #ccc; font-size:13px; line-height:1.8;">
        <b>🗺️ {village.capitalize()}</b><br>
        <span style="color:#1565C0">●</span> Hydrant (n={len(hydrants)})<br>
        <span style="color:#2E7D32">●</span> Candidate Pool / AI (n={len(pools)})<br>
        <span style="background:#FFD700; padding:0 6px;">  </span> Coverage Gain (+{df_results[df_results.village==village.capitalize()].gain_pct.values[0]}%)<br>
        <i style="font-size:11px">Buffer-Radius: {CONFIG["BUFFER_RADIUS"]}m</i>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    folium.LayerControl(collapsed=False).add_to(m)

    #Save the mpas
    map_path = OUTPUT_DIR / f"map_{village}.html"
    m.save(str(map_path))
    print(f"Saved to: {map_path}")



  Erstelle Karte für velden...
Saved to: outputs\map_velden.html

  Erstelle Karte für hermagor...
Saved to: outputs\map_hermagor.html

  Erstelle Karte für treffen...
Saved to: outputs\map_treffen.html
